In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [12]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. 이미지 변환 설정 (Food-101용)
# 모델 입력 크기를 맞추기 위해 64x64로 줄이는 것이 좋습니다.
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 2. Food-101 데이터셋 다운로드 및 로드
# 이 부분이 질문하신 코드의 대체재입니다.
training_data = datasets.Food101(root="data", split="train", download=True, transform=transform)
test_data = datasets.Food101(root="data", split="test", download=True, transform=transform)

# 3. 데이터로더 생성 (배치 학습을 위해 필수)
batch_size = 64
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

100%|██████████| 5.00G/5.00G [13:25<00:00, 6.20MB/s]   


In [13]:
batch_size = 64

# 데이터로더를 생성합니다.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 3, 64, 64])
Shape of y: torch.Size([64]) torch.int64


In [14]:
from torch import nn

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(64*64*3, 512), 
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 101) # 101개 카테고리
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.linear_relu_stack(x)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=12288, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=101, bias=True)
  )
)


In [15]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [16]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # 예측 오류 계산
        pred = model(X)
        loss = loss_fn(pred, y)

        # 역전파
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [17]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [34]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. 이미지 변환 설정 (Food-101용)
# 모델 입력 크기를 맞추기 위해 64x64로 줄이는 것이 좋습니다.
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 2. Food-101 데이터셋 다운로드 및 로드
# 이 부분이 질문하신 코드의 대체재입니다.
training_data = datasets.Food101(root="data", split="train", download=True, transform=transform)
test_data = datasets.Food101(root="data", split="test", download=True, transform=transform)

# 3. 데이터로더 생성 (배치 학습을 위해 필수)
batch_size = 64
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

100%|██████████| 5.00G/5.00G [24:55<00:00, 3.34MB/s]   


In [35]:
batch_size = 64

# 데이터로더를 생성합니다.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 3, 64, 64])
Shape of y: torch.Size([64]) torch.int64


In [36]:
# 학습에 사용할 CPU나 GPU, MPS 장치를 얻습니다.
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

# 모델을 정의합니다.
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using cuda device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [37]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [41]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # 예측 오류 계산
        pred = model(X)
        loss = loss_fn(pred, y)

        # 역전파
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [42]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [44]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            # 28*28(784) 대신 Food101 이미지 크기인 3*224*224(12288) 입력!
            nn.Linear(3 * 224 * 224, 512), 
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            # FashionMNIST의 10개 클래스 대신 Food101의 101개 클래스로 변경!
            nn.Linear(512, 101) 
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

# 수정된 구조로 모델 객체 다시 생성
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=150528, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=101, bias=True)
  )
)


In [45]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [46]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # 예측 오류 계산
        pred = model(X)
        loss = loss_fn(pred, y)

        # 역전파
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [47]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [48]:
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------


RuntimeError: mat1 and mat2 shapes cannot be multiplied (64x12288 and 150528x512)

In [49]:
import torch
from torch import nn

# 학습에 사용할 CPU나 GPU, MPS 장치를 얻습니다.
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

# 모델을 정의합니다. (Food101 데이터 크기인 12288과 클래스 101개에 맞춤)
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            # 들어오는 데이터 크기 12288을 첫 번째 인자로 넣어줍니다.
            nn.Linear(12288, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            # Food101의 클래스 개수인 101로 최종 출력 설정
            nn.Linear(512, 101)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

# 모델을 생성하고 장치로 이동
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=12288, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=101, bias=True)
  )
)


In [50]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [51]:
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 4.654482  [   64/75750]
loss: 4.602472  [ 6464/75750]
loss: 4.649473  [12864/75750]
loss: 4.568443  [19264/75750]
loss: 4.628378  [25664/75750]
loss: 4.601116  [32064/75750]
loss: 4.660072  [38464/75750]
loss: 4.594046  [44864/75750]
loss: 4.793958  [51264/75750]
loss: 4.575320  [57664/75750]
loss: 4.770371  [64064/75750]
loss: 4.543487  [70464/75750]
Test Error: 
 Accuracy: 1.8%, Avg loss: 4.604696 

Epoch 2
-------------------------------
loss: 4.688709  [   64/75750]
loss: 4.637400  [ 6464/75750]
loss: 4.690789  [12864/75750]
loss: 4.598370  [19264/75750]
loss: 4.656900  [25664/75750]
loss: 4.616809  [32064/75750]
loss: 4.632353  [38464/75750]
loss: 4.473679  [44864/75750]
loss: 4.716569  [51264/75750]
loss: 4.521855  [57664/75750]
loss: 4.790092  [64064/75750]
loss: 4.463078  [70464/75750]
Test Error: 
 Accuracy: 2.0%, Avg loss: 4.591998 

Epoch 3
-------------------------------
loss: 4.752705  [   64/75750]
loss: 4.690845  [ 6464/75750

In [3]:
import torch
import torch.nn as nn

# 1. ResNet의 핵심인 기본 블록(Basic Block) 구현
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        # 첫 번째 합성곱 레이어
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        # 두 번째 합성곱 레이어
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 지름길(Shortcut / Skip Connection) 설정
        self.shortcut = nn.Sequential()
        # 만약 입력 채널과 출력 채널 크기가 다르거나, 이미지 해상도(stride)가 바뀌면 크기를 맞춰줍니다.
        if stride != 1 or in_channels != self.expansion * out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, self.expansion * out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # ⭐️ 잔차 연결: 원래 입력(identity)을 연산 결과(out)에 더해줍니다!
        out += identity
        out = self.relu(out)
        return out


# 2. ResNet34 모델 클래스 정의
class ResNet34(nn.Module):
    def __init__(self, block, layers, num_classes=101): # Food101 클래스 개수 101개
        super(ResNet34, self).__init__()
        self.in_channels = 64

        # 입력층 (처음 이미지를 받아들이는 부분)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet34 구조에 맞는 4개의 Layer 스테이지 생성 ([3, 4, 6, 3] 블록)
        self.layer1 = self._make_layer(block, 64, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        # 출력층 (평균 풀링 및 최종 분류기)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride):
        strides = [stride] + [1] * (blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_channels, out_channels, s))
            self.in_channels = out_channels * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# 3. ResNet34 인스턴스 생성 및 장치(GPU) 이동
# [3, 4, 6, 3]은 ResNet34 표준에 정의된 각 스테이지별 블록 개수입니다.
resnet_model = ResNet34(BasicBlock, [3, 4, 6, 3], num_classes=101).to(device)
print("=== ResNet34 모델 구조 준비 완료 ===")
print(resnet_model)

# 4. 동일한 환경 세팅 (손실함수 및 SGD 옵티마이저)
loss_fn = nn.CrossEntropyLoss()
# 베이스라인과 동일하게 SGD를 사용하고, 모델 매개변수만 새로운 레즈넷으로 연결합니다.
resnet_optimizer = torch.optim.SGD(resnet_model.parameters(), lr=1e-3)

# 5. ResNet34 학습 시작 (10 에폭)
epochs = 10
print("\n=== ResNet34 학습을 시작합니다 ===")
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    # 위에서 정의되어 있던 train, test 함수를 그대로 재활용하되 모델과 옵티마이저만 바꿔 넣습니다.
    train(train_dataloader, resnet_model, loss_fn, resnet_optimizer)
    test(test_dataloader, resnet_model, loss_fn)
print("ResNet34 Training Done!")

NameError: name 'device' is not defined